# Autoregressive Text Autocomplete Engine using LSTM

## Training Notebook for Google Colab

This notebook trains a word-level LSTM language model on WikiText-2 for text autocompletion.

### Key Concepts Covered:
- Word-level language modeling
- Teacher forcing during training
- LSTM architecture for sequence generation
- Vocabulary management and OOV handling

### Setup Instructions:
1. Open in Google Colab
2. Runtime → Change runtime type → GPU
3. Run all cells sequentially
4. Download saved model files

---

## 1. Environment Setup

First, we verify GPU availability and install the HuggingFace datasets library for WikiText-2 access.

In [ ]:
# Check GPU availability
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# Install HuggingFace datasets
!pip install datasets -q
print("✓ Environment ready")

## 2. Imports and Configuration

### Why These Configurations?

| Parameter | Value | Rationale |
|-----------|-------|----------|
| `VOCAB_SIZE=20000` | Top 20K words cover ~95% of typical English text |
| `SEQUENCE_LENGTH=30` | Balances context length vs. computational cost |
| `EMBEDDING_DIM=128` | Sufficient for word semantics without overfitting |
| `LSTM_UNITS=256` | Good capacity for language patterns |
| `BATCH_SIZE=64` | Optimal for Colab GPU memory |

In [ ]:
import numpy as np
import pickle
import json
import re
from tqdm.auto import tqdm

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from datasets import load_dataset

# ============ CONFIGURATION ============
# Model Architecture
VOCAB_SIZE = 20000       # Maximum vocabulary size
SEQUENCE_LENGTH = 30     # Input sequence length
EMBEDDING_DIM = 128      # Word embedding dimensions
LSTM_UNITS = 256         # LSTM hidden units
DROPOUT_RATE = 0.3       # Dropout for regularization

# Training
BATCH_SIZE = 64          # Batch size for training
EPOCHS = 20              # Maximum training epochs
LEARNING_RATE = 0.001    # Adam optimizer learning rate

# Paths
MODEL_SAVE_PATH = 'autocomplete_lstm.h5'
TOKENIZER_SAVE_PATH = 'tokenizer.pkl'
CONFIG_SAVE_PATH = 'config.json'

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✓ Configuration set")

## 3. Load WikiText-2 Dataset

### Why WikiText-2?
- **Clean Wikipedia text**: High-quality, grammatically correct English
- **Manageable size**: ~2M tokens, trains quickly on Colab GPU
- **Standard benchmark**: Commonly used for language modeling research
- **Long-form content**: Contains coherent articles, not just sentences

In [ ]:
# Load WikiText-2 from HuggingFace
print("Loading WikiText-2 dataset...")
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')

# Extract text from splits
train_texts = dataset['train']['text']
val_texts = dataset['validation']['text']
test_texts = dataset['test']['text']

print(f"\nDataset Statistics:")
print(f"  Training samples: {len(train_texts):,}")
print(f"  Validation samples: {len(val_texts):,}")
print(f"  Test samples: {len(test_texts):,}")

# Preview samples
print(f"\nSample text:")
for i, text in enumerate(train_texts[10:13]):
    if text.strip():
        print(f"  [{i}] {text[:100]}...")

## 4. Text Preprocessing

### Preprocessing Pipeline:
1. **Clean text**: Remove special characters, normalize whitespace
2. **Filter empty lines**: Remove blank entries from dataset
3. **Tokenization**: Convert words to integer indices
4. **Vocabulary capping**: Keep only top VOCAB_SIZE words

### Why Word-Level (Not Character-Level)?
- Captures semantic meaning directly
- Shorter sequences (30 words vs 150+ characters)
- Better for sentence-level generation
- More interpretable outputs

In [ ]:
def clean_text(text):
    """Clean and normalize text."""
    if not isinstance(text, str):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Remove special characters (keep basic punctuation)
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\!\?\'\-]', '', text)
    
    # Normalize punctuation
    text = re.sub(r'\.+', '.', text)
    text = re.sub(r'\,+', ',', text)
    
    return text.strip()

# Clean all texts
print("Cleaning texts...")
train_texts_clean = [clean_text(t) for t in tqdm(train_texts) if clean_text(t)]
val_texts_clean = [clean_text(t) for t in tqdm(val_texts) if clean_text(t)]

print(f"\nCleaned texts:")
print(f"  Training: {len(train_texts_clean):,}")
print(f"  Validation: {len(val_texts_clean):,}")

# Combine for tokenizer fitting
all_texts = train_texts_clean + val_texts_clean
print(f"  Total for tokenization: {len(all_texts):,}")

## 5. Build Tokenizer & Vocabulary

### Tokenizer Configuration:
- `num_words=VOCAB_SIZE`: Limits vocabulary to top N most frequent words
- `oov_token='<unk>'`: Unknown words are mapped to this token
- `lower=True`: All text converted to lowercase
- `filters`: Characters to remove (punctuation handled separately)

In [ ]:
# Initialize tokenizer
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token='<unk>',
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
    lower=True
)

# Fit on training data
print("Fitting tokenizer...")
tokenizer.fit_on_texts(all_texts)

# Vocabulary statistics
total_words = len(tokenizer.word_index)
actual_vocab_size = min(VOCAB_SIZE, total_words + 1)

print(f"\nVocabulary Statistics:")
print(f"  Total unique words found: {total_words:,}")
print(f"  Vocabulary size (capped): {actual_vocab_size:,}")

# Show most common words
print(f"\nTop 20 most common words:")
word_counts = sorted(tokenizer.word_counts.items(), key=lambda x: x[1], reverse=True)
for word, count in word_counts[:20]:
    print(f"  {word}: {count:,}")

## 6. Create Training Sequences

### Sequence Generation Process:

For language modeling, we predict the next word given previous context:
```
Input:  [word_1, word_2, word_3, ..., word_n-1]
Output: word_n
```

We use a **sliding window** approach:
```
Text: "the cat sat on the mat"

Sample 1: Input=[the, cat, sat, on, the] → Output=mat
Sample 2: Input=[cat, sat, on, the, mat] → Output=.
```

### TEACHER FORCING
During training, each input uses **ground-truth previous words**, not model predictions.

**Advantage**: Faster, stable training  
**Disadvantage**: Creates **exposure bias** - model never sees its own errors during training

In [ ]:
def create_sequences(texts, tokenizer, sequence_length):
    """
    Create input-output pairs for language modeling.
    
    Uses sliding window to create sequences where:
    - X = previous (sequence_length) tokens
    - y = next token to predict
    """
    # Convert texts to sequences
    sequences = tokenizer.texts_to_sequences(texts)
    
    # Flatten into continuous token stream
    all_tokens = []
    for seq in sequences:
        all_tokens.extend(seq)
    
    print(f"Total tokens: {len(all_tokens):,}")
    
    # Create sliding window sequences
    X, y = [], []
    
    for i in tqdm(range(sequence_length, len(all_tokens))):
        # Input: previous sequence_length tokens
        input_seq = all_tokens[i - sequence_length:i]
        # Output: next token
        output_token = all_tokens[i]
        
        X.append(input_seq)
        y.append(output_token)
    
    return np.array(X), np.array(y)

# Create training sequences
print("Creating training sequences...")
X_train, y_train = create_sequences(train_texts_clean, tokenizer, SEQUENCE_LENGTH)

print(f"\nTraining sequences:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  Memory: {X_train.nbytes / 1024 / 1024:.1f} MB")

# Create validation sequences
print("\nCreating validation sequences...")
X_val, y_val = create_sequences(val_texts_clean, tokenizer, SEQUENCE_LENGTH)

print(f"\nValidation sequences:")
print(f"  X_val shape: {X_val.shape}")
print(f"  y_val shape: {y_val.shape}")

## 7. Build LSTM Model

### Architecture Design Decisions:

**1. Embedding Layer**
- Maps integer word indices to dense vectors
- Learned during training (not pretrained)
- 128 dimensions balances expressiveness vs computation

**2. Stacked LSTM (2 layers)**
- First LSTM returns sequences for stacking
- Second LSTM returns only final hidden state
- 256 units provides sufficient capacity
- Stacking allows hierarchical temporal pattern learning

**3. Dropout (0.3)**
- Applied after each LSTM layer
- Prevents overfitting on training data

**4. Dense Softmax Output**
- Outputs probability distribution over vocabulary
- Each output is P(word_i | context)

### Why No Attention?
This project deliberately uses pure LSTM to demonstrate:
- How RNNs model sequences without position-independent attention
- Limitations of fixed-length context
- Error accumulation in recurrent processing

In [ ]:
def build_lstm_model(vocab_size, sequence_length, embedding_dim, lstm_units, dropout_rate):
    """
    Build the LSTM language model.
    
    Architecture:
    Input (seq_len,) → Embedding → LSTM → Dropout → LSTM → Dropout → Dense (softmax)
    """
    model = Sequential([
        # Embedding: word index → dense vector
        Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            input_length=sequence_length,
            name='embedding'
        ),
        
        # First LSTM layer - returns full sequence
        LSTM(
            units=lstm_units,
            return_sequences=True,  # Pass to next LSTM
            name='lstm_1'
        ),
        Dropout(dropout_rate, name='dropout_1'),
        
        # Second LSTM layer - returns final state
        LSTM(
            units=lstm_units,
            return_sequences=False,  # Only final hidden state
            name='lstm_2'
        ),
        Dropout(dropout_rate, name='dropout_2'),
        
        # Output: probability over vocabulary
        Dense(
            units=vocab_size,
            activation='softmax',
            name='output'
        )
    ])
    
    return model

# Build model
model = build_lstm_model(
    vocab_size=actual_vocab_size,
    sequence_length=SEQUENCE_LENGTH,
    embedding_dim=EMBEDDING_DIM,
    lstm_units=LSTM_UNITS,
    dropout_rate=DROPOUT_RATE
)

# Compile with sparse categorical crossentropy
# (labels are integers, not one-hot)
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summary
print("Model Architecture:")
model.summary()

## 8. Training Callbacks

### Callback Configuration:

**EarlyStopping**: Stop training if validation loss doesn't improve for 3 epochs
- Prevents overfitting
- Saves training time

**ModelCheckpoint**: Save best model based on validation loss
- Ensures we keep the best weights
- Guards against training degradation

**ReduceLROnPlateau**: Reduce learning rate when loss plateaus
- Helps escape local minima
- Enables finer optimization in later epochs

In [ ]:
# Define callbacks
callbacks = [
    # Stop if no improvement for 3 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Save best model
    ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    
    # Reduce LR on plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("✓ Callbacks configured")

## 9. Train the Model

### Training Process:
1. Feed batches of (input_sequence, next_word) pairs
2. Model predicts probability distribution over vocabulary
3. Calculate cross-entropy loss against true next word
4. Backpropagate gradients through LSTM
5. Update weights with Adam optimizer

### Expected Training Time:
- **Colab GPU (T4)**: ~30-45 minutes
- **Colab Pro (A100)**: ~15-20 minutes

### Monitoring:
- **Loss**: Should decrease steadily
- **Accuracy**: Will be low (predicting from 20K classes)
- **val_loss**: Should track train loss (watch for overfitting)

In [ ]:
# Train model
print("="*60)
print("TRAINING STARTED")
print("="*60)
print(f"Training samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max epochs: {EPOCHS}")
print("="*60)

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)

## 10. Training History Visualization

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

# Print final metrics
print(f"\nFinal Training Loss: {history.history['loss'][-1]:.4f}")
print(f"Final Validation Loss: {history.history['val_loss'][-1]:.4f}")
print(f"Final Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

## 11. Save Model & Tokenizer

### Files Saved:
1. **autocomplete_lstm.h5**: Keras model with trained weights
2. **tokenizer.pkl**: Fitted tokenizer for text processing
3. **config.json**: Model configuration for reproducibility

These files are required for the Streamlit deployment.

In [ ]:
# Save model (already saved by ModelCheckpoint, but explicit save)
model.save(MODEL_SAVE_PATH)
print(f"✓ Model saved to: {MODEL_SAVE_PATH}")

# Save tokenizer
tokenizer_data = {
    'tokenizer': tokenizer,
    'vocab_size': VOCAB_SIZE,
    'sequence_length': SEQUENCE_LENGTH,
    'oov_token': '<unk>',
    'is_fitted': True
}
with open(TOKENIZER_SAVE_PATH, 'wb') as f:
    pickle.dump(tokenizer_data, f)
print(f"✓ Tokenizer saved to: {TOKENIZER_SAVE_PATH}")

# Save configuration
config = {
    'vocab_size': VOCAB_SIZE,
    'sequence_length': SEQUENCE_LENGTH,
    'embedding_dim': EMBEDDING_DIM,
    'lstm_units': LSTM_UNITS,
    'dropout_rate': DROPOUT_RATE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'actual_vocab_size': actual_vocab_size
}
with open(CONFIG_SAVE_PATH, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✓ Config saved to: {CONFIG_SAVE_PATH}")

## 12. Test Generation (Inference)

### AUTOREGRESSIVE GENERATION vs TRAINING

**Training (Teacher Forcing):**
- Model receives ground-truth tokens at each step
- Never sees its own predictions
- Creates **exposure bias**

**Inference (Autoregressive):**
- Model feeds its own predictions back as input
- Errors can compound over time
- Requires careful decoding strategies

### DECODING STRATEGIES:

| Strategy | Description | Behavior |
|----------|-------------|----------|
| Greedy | Always pick argmax | Deterministic, safe |
| Sampling | Sample from distribution | Creative, diverse |
| Temperature | Scale logits before softmax | Control randomness |

### TEMPERATURE SCALING:
```
P'(word_i) = softmax(logit_i / T)

T < 1.0 → Sharper distribution (more confident)
T = 1.0 → Original distribution  
T > 1.0 → Flatter distribution (more random)
```

In [ ]:
def apply_temperature(logits, temperature):
    """Apply temperature scaling to logits."""
    if temperature <= 0:
        raise ValueError("Temperature must be positive")
    
    scaled = logits / temperature
    scaled = scaled - np.max(scaled)  # Numerical stability
    exp_logits = np.exp(scaled)
    return exp_logits / np.sum(exp_logits)


def generate_text(model, tokenizer, seed_text, max_tokens=20, temperature=0.8, stop_at_period=True):
    """
    Generate text continuation autoregressively.
    
    The Autoregressive Loop:
    1. Tokenize seed text
    2. Pad to sequence_length
    3. Predict next token
    4. Append prediction to sequence
    5. Repeat until stop condition
    """
    # Tokenize and pad
    sequence = tokenizer.texts_to_sequences([seed_text.lower()])[0]
    sequence = pad_sequences([sequence], maxlen=SEQUENCE_LENGTH, padding='pre')[0]
    current_sequence = np.expand_dims(sequence, axis=0)
    
    generated_words = []
    reverse_word_index = {v: k for k, v in tokenizer.word_index.items()}
    
    for _ in range(max_tokens):
        # Predict next token
        predictions = model.predict(current_sequence, verbose=0)[0]
        
        # Apply temperature and sample
        probs = apply_temperature(predictions, temperature)
        token_index = np.random.choice(len(probs), p=probs)
        
        # Skip padding/unknown
        if token_index == 0:
            continue
        
        # Decode token
        word = reverse_word_index.get(token_index, '<unk>')
        generated_words.append(word)
        
        # Check stop condition
        if stop_at_period and word in ['.', '!', '?']:
            break
        
        # Update sequence (shift left, append new token)
        current_sequence = np.roll(current_sequence, -1, axis=1)
        current_sequence[0, -1] = token_index
    
    # Post-process
    result = ' '.join(generated_words)
    for punct in ['.', ',', '!', '?']:
        result = result.replace(f' {punct}', punct)
    
    return result


print("\n" + "="*60)
print("GENERATION TEST")
print("="*60)

### Test Generation with Different Temperatures

In [ ]:
# Test prompts
test_prompts = [
    "artificial intelligence is transforming",
    "the history of science shows that",
    "in the modern world education",
    "climate change is affecting"
]

# Test different temperatures
temperatures = [0.5, 0.8, 1.0, 1.3]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: '{prompt}'")
    print(f"{'='*60}")
    
    for temp in temperatures:
        generated = generate_text(model, tokenizer, prompt, temperature=temp)
        print(f"\n[T={temp}] {generated}")

## 13. Understanding Exposure Bias

### What is Exposure Bias?

**During Training (Teacher Forcing):**
- Model always receives **ground-truth** previous tokens
- Never sees its own mistakes
- Learns to predict: P(word_n | perfect_context)

**During Inference (Autoregressive):**
- Model receives its **own predictions** as context
- Errors from early predictions affect later ones
- Actually predicts: P(word_n | noisy_context)

### Effects of Exposure Bias:
1. **Error Accumulation**: Small errors compound over time
2. **Semantic Drift**: Text becomes less coherent over length
3. **Repetition Loops**: Model may get stuck in patterns

### Mitigation Strategies (Implemented):
- Temperature scaling
- Maximum token limits (20 tokens)
- Stop at sentence boundaries

### Advanced Strategies (Not Implemented):
- Scheduled Sampling
- Beam Search
- Nucleus (Top-p) Sampling

In [ ]:
# Demonstrate error accumulation
print("DEMONSTRATING ERROR ACCUMULATION")
print("="*60)
print("Generating with increasing max_tokens to show degradation:")

prompt = "the development of technology has"
for max_tokens in [5, 10, 20, 40]:
    generated = generate_text(
        model, tokenizer, prompt, 
        max_tokens=max_tokens, 
        temperature=0.8,
        stop_at_period=False  # Force full generation
    )
    print(f"\n[max={max_tokens}] {generated}")

print("\n" + "="*60)
print("Note: Longer generations often show more incoherence.")
print("This is the effect of error accumulation in autoregressive models.")

## 14. Download Files for Deployment

Download the three files needed for the Streamlit app:
1. `autocomplete_lstm.h5` - Trained model
2. `tokenizer.pkl` - Fitted tokenizer
3. `config.json` - Model configuration

Place these files in the `models/` directory of your project.

In [ ]:
from google.colab import files
import os

# List saved files
print("Files to download:")
for filepath in [MODEL_SAVE_PATH, TOKENIZER_SAVE_PATH, CONFIG_SAVE_PATH]:
    size = os.path.getsize(filepath) / 1024 / 1024  # MB
    print(f"  {filepath}: {size:.2f} MB")

# Create zip for easier download
!zip -r model_files.zip {MODEL_SAVE_PATH} {TOKENIZER_SAVE_PATH} {CONFIG_SAVE_PATH}

print("\nDownloading model_files.zip...")
files.download('model_files.zip')

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print("\nNext steps:")
print("1. Extract model_files.zip")
print("2. Place files in models/ directory")
print("3. Run: streamlit run app/streamlit_app.py")

---

## Summary

### What We Built:
- Word-level LSTM language model (no transformers, no attention)
- Trained on WikiText-2 (~2M tokens)
- 20K vocabulary with OOV handling
- Autoregressive generation with temperature scaling

### Key Concepts Demonstrated:
1. **Teacher Forcing**: Training with ground-truth context
2. **Exposure Bias**: Train/inference distribution mismatch
3. **Temperature Scaling**: Controlling generation randomness
4. **Error Accumulation**: Degradation in long generations

### Files Generated:
- `autocomplete_lstm.h5`: Trained model weights
- `tokenizer.pkl`: Fitted vocabulary/tokenizer
- `config.json`: Model configuration

### Next: Deploy with Streamlit
See `app/streamlit_app.py` for the web interface implementation.